In [ ]:
!pip install SpeechRecognition

In [ ]:
from google.colab import output
import speech_recognition as sr
import io
from base64 import b64decode

# Install pydub (for audio conversion) and ffmpeg (required by pydub for format handling)
!pip install pydub
!apt-get install -y ffmpeg

from pydub import AudioSegment

# Initialize the speech recognizer
recognizer = sr.Recognizer()

# Enable microphone access in Colab (requires user permission)
print("Please allow microphone access when prompted")
# JavaScript code executed in the browser to record audio from microphone
audio_string = output.eval_js('''
  async function getAudio() {
    // Request access to user's microphone
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    // Create a MediaRecorder to capture audio from the stream
    const mediaRecorder = new MediaRecorder(stream);
    const audioChunks = [];

    // Event handler that runs when audio data is available
    mediaRecorder.ondataavailable = event => {
      audioChunks.push(event.data);  // Store each chunk of audio data
    };

    // Start recording
    mediaRecorder.start();
    // Wait for 5 seconds while recording
    await new Promise(resolve => setTimeout(resolve, 5000)); // Record for 5 seconds
    // Stop recording
    mediaRecorder.stop();

    // Wait for the recording to fully stop
    await new Promise(resolve => mediaRecorder.onstop = resolve);
    // Create a Blob (binary large object) from the recorded chunks in WebM format
    const audioBlob = new Blob(audioChunks, { 'type' : 'audio/webm' }); // Specify type as webm
    // Create a FileReader to read the Blob as a data URL
    const reader = new FileReader();

    // Return a Promise that resolves with the base64-encoded audio data
    return new Promise(resolve => {
      // When reading is complete, split off the data URL prefix and keep only the base64 part
      reader.onloadend = () => resolve(reader.result.split(',')[1]);
      // Read the blob as a data URL (base64 encoded string)
      reader.readAsDataURL(audioBlob);
    });
  }
  // Execute the function and return the base64 audio data
  getAudio()
''')

# Decode the base64 string back into binary audio data
audio_bytes = b64decode(audio_string)

# Save the binary audio data to a temporary WebM file
with open('temp_audio.webm', 'wb') as f:
    f.write(audio_bytes)

# Convert the WebM audio file to WAV format using pydub (required for speech_recognition)
webm_audio = AudioSegment.from_file('temp_audio.webm', format='webm')
webm_audio.export('temp_audio.wav', format='wav')

# Use the speech recognition library on the converted WAV file
with sr.AudioFile('temp_audio.wav') as source:
    print("Processing your speech...")
    # Record the audio data from the file
    audio_data = recognizer.record(source)
    try:
        # Send the audio to Google's speech recognition API
        text = recognizer.recognize_google(audio_data)
        print("You said:", text)
    except sr.UnknownValueError:
        # Error when Google Speech Recognition can't understand the audio
        print("Google Speech Recognition could not understand audio")
    except sr.RequestError as e:
        # Error when there's a problem connecting to the Google service
        print(f"Could not request results from Google Speech Recognition service; {e}")

# Clean up temporary files
!rm temp_audio.webm
!rm temp_audio.wav


In [ ]:
!pip install gtts

In [ ]:
# Import required libraries
from gtts import gTTS  # Google Text-to-Speech library for converting text to speech
from IPython.display import Audio, display  # For playing audio in Jupyter/Colab notebooks
import os  # Operating system interface (though not used in this code)

# ===== ENGLISH TEXT-TO-SPEECH GENERATION =====
# Define the text to be converted to speech in English
text = "Hello, this is a test."

# Create a gTTS object with English language settings
# - text: The text content to convert to speech
# - lang: Language code "en" for English
tts = gTTS(text=text, lang="en")

# Save the generated speech as an MP3 file
tts.save("output.mp3")

# Display an audio player in the notebook to play the generated English speech
display(Audio("output.mp3"))


# ===== FRENCH TEXT-TO-SPEECH GENERATION =====
# Create a French version with slower speech rate
# - text: French phrase "Bonjour" meaning "Hello"
# - lang: Language code "fr" for French
# - slow=True: Speaks the text at a slower pace
tts_fr = gTTS(text="Bonjour", lang="fr", slow=True)

# Save the French speech to a different file to avoid overwriting the English version
tts_fr.save("output_fr.mp3")

# Display an audio player for the French speech in the notebook
display(Audio("output_fr.mp3"))
